# Object Tracking — 4 Videos (FIXED)
## Harris Corners + Lucas–Kanade Optical Flow + Color-Guided Re-detection

| Section | Video | Object | Key Fix |
|---------|-------|--------|---------|
| 1 | `19-May-2026_at_2_12_PM.mov` | Rolling ball (near wall) | Correct ROI on the ball, not the wall texture |
| 2 | `19-May-2026_at_2_13_PM.mov` | Ball rolling (courtyard) | Correct entry ROI at bottom of frame |
| 3 | `19-May-2026_at_2_15_PM.mov` | Person in white shalwar kameez | White-clothing color lock + capped bbox growth to exclude other people |
| 4 | `19-May-2026_at_2_16_PM.MOV` | Eagle/hawk flying in open sky | Correct ROI on the lower open-sky bird, not the small bird near wire |

**Run all cells top-to-bottom.**


In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ── LK / Harris parameters ──────────────────────────────────────────────────
HARRIS_K        = 0.04
LK_WIN_SIZE     = (21, 21)
LK_MAX_LEVEL    = 3
LK_CRITERIA     = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
FB_ERROR_THRESH = 1.5
MIN_POINTS      = 4
BOX_COLOR       = (0, 255, 0)
POINT_COLOR     = (0, 0, 255)
TRAIL_COLOR     = (255, 200, 0)
TRAIL_LEN       = 20

# ────────────────────────────────────────────────────────────────────────────
def detect_harris_corners(gray, roi, block_size=3, ksize=3,
                          thresh_ratio=0.01, min_dist=8, max_corners=200):
    x, y, w, h = roi
    fh, fw = gray.shape[:2]
    x = max(0, x); y = max(0, y)
    w = min(w, fw - x); h = min(h, fh - y)
    if w <= 0 or h <= 0:
        return np.empty((0, 1, 2), dtype=np.float32)
    patch = gray[y:y+h, x:x+w].astype(np.float32)
    R = cv2.cornerHarris(patch, block_size, ksize, HARRIS_K)
    R = cv2.dilate(R, None)
    if R.max() == 0:
        return np.empty((0, 1, 2), dtype=np.float32)
    mask = R > thresh_ratio * R.max()
    ys, xs = np.where(mask)
    order = np.argsort(-R[ys, xs])
    ys, xs = ys[order], xs[order]
    selected = []; occ = np.zeros(R.shape, bool)
    for cy, cx in zip(ys, xs):
        if occ[cy, cx]: continue
        selected.append((cx + x, cy + y))
        r0 = max(0, cy - min_dist); r1 = min(R.shape[0], cy + min_dist + 1)
        c0 = max(0, cx - min_dist); c1 = min(R.shape[1], cx + min_dist + 1)
        occ[r0:r1, c0:c1] = True
        if len(selected) >= max_corners: break
    if not selected:
        return np.empty((0, 1, 2), dtype=np.float32)
    return np.array(selected, dtype=np.float32).reshape(-1, 1, 2)


def track_lk(prev_gray, curr_gray, prev_pts):
    if prev_pts is None or len(prev_pts) == 0:
        return np.empty((0, 1, 2), dtype=np.float32), np.empty((0, 1, 2), dtype=np.float32)
    lk = dict(winSize=LK_WIN_SIZE, maxLevel=LK_MAX_LEVEL, criteria=LK_CRITERIA)
    nxt, sf, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None, **lk)
    bck, sb, _ = cv2.calcOpticalFlowPyrLK(curr_gray, prev_gray, nxt, None, **lk)
    err = np.linalg.norm(prev_pts.reshape(-1, 2) - bck.reshape(-1, 2), axis=1)
    ok = (sf.ravel() == 1) & (sb.ravel() == 1) & (err < FB_ERROR_THRESH)
    return nxt[ok], prev_pts[ok]


def pts_to_bbox(pts, pad=10):
    xy = pts.reshape(-1, 2); x1, y1 = xy.min(0); x2, y2 = xy.max(0)
    return (max(0, int(x1) - pad), max(0, int(y1) - pad),
            int(x2 - x1) + 2 * pad, int(y2 - y1) + 2 * pad)


def pts_to_centroid(pts):
    xy = pts.reshape(-1, 2)
    return int(xy[:, 0].mean()), int(xy[:, 1].mean())


def clamp(bbox, shape):
    fh, fw = shape[:2]; x, y, w, h = bbox
    x = max(0, min(x, fw - 1)); y = max(0, min(y, fh - 1))
    w = max(1, min(w, fw - x)); h = max(1, min(h, fh - y))
    return x, y, w, h


def annotate(frame, bbox, pts, trail, fidx, label, active=True):
    vis = frame.copy(); x, y, w, h = bbox
    for pp in trail:
        for p in pp:
            cv2.circle(vis, (int(p[0]), int(p[1])), 1, TRAIL_COLOR, -1)
    for pt in pts.reshape(-1, 2):
        cv2.circle(vis, (int(pt[0]), int(pt[1])), 3, POINT_COLOR, -1)
    cv2.rectangle(vis, (x, y), (x + w, y + h), BOX_COLOR if active else (0, 0, 200), 2)
    if len(pts) > 0:
        cx, cy = pts_to_centroid(pts)
        cv2.drawMarker(vis, (cx, cy), (255, 255, 0), cv2.MARKER_CROSS, 15, 2)
    st = "TRACKING" if active else "SEARCHING"
    cv2.putText(vis, f"{label} | F{fidx} | {st} | pts:{len(pts)}",
                (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    return vis


def run_tracker(input_path, output_path, roi, label="",
                start_frame=0, thresh_ratio=0.01, block_size=3, min_dist=8,
                max_bbox_expand=None):
    """
    Standard Harris+LK tracker.
    max_bbox_expand: if set, limits how much the bounding box can grow each frame
                     (prevents the box from latching onto nearby objects).
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open: {input_path}")
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    fw     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh_v   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = cv2.VideoWriter(output_path,
                          cv2.VideoWriter_fourcc(*"mp4v"), fps, (fw, fh_v))
    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    ret, frame = cap.read()
    if not ret:
        cap.release(); out.release()
        raise RuntimeError("Cannot read first frame.")
    prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    bbox = clamp(roi, frame.shape)
    prev_pts = detect_harris_corners(prev_gray, bbox,
                                     block_size=block_size,
                                     thresh_ratio=thresh_ratio,
                                     min_dist=min_dist)
    if len(prev_pts) == 0:
        cap.release(); out.release()
        raise RuntimeError(f"[{label}] No corners in ROI — try a larger box.")
    print(f"[{label}] Start frame={start_frame}, corners={len(prev_pts)}, roi={roi}")
    trail = []; fidx = start_frame; active = True
    metrics = {"frame": [], "n_pts": [], "bbox": [], "centroid": [], "lost": 0, "redet": 0}
    out.write(annotate(frame, bbox, prev_pts, [], fidx, label))
    metrics["frame"].append(fidx); metrics["n_pts"].append(len(prev_pts))
    metrics["bbox"].append(bbox); metrics["centroid"].append(pts_to_centroid(prev_pts))
    fidx += 1
    while True:
        ret, frame = cap.read()
        if not ret: break
        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        good_new, _ = track_lk(prev_gray, curr_gray, prev_pts)
        survived = len(good_new)
        if survived >= MIN_POINTS:
            new_bbox = clamp(pts_to_bbox(good_new), frame.shape)
            # Optionally cap bbox growth to prevent drifting onto other objects
            if max_bbox_expand is not None:
                bx, by, bw, bh = bbox
                nx, ny, nw, nh = new_bbox
                cx_old, cy_old = bx + bw // 2, by + bh // 2
                cx_new, cy_new = nx + nw // 2, ny + nh // 2
                # Limit size to original + max_bbox_expand
                nw = min(nw, bw + max_bbox_expand)
                nh = min(nh, bh + max_bbox_expand)
                new_bbox = clamp((nx, ny, nw, nh), frame.shape)
            bbox = new_bbox
            prev_pts = good_new; active = True
        else:
            metrics["redet"] += 1
            new_pts = detect_harris_corners(curr_gray, bbox,
                                            block_size=block_size,
                                            thresh_ratio=thresh_ratio,
                                            min_dist=min_dist)
            if len(new_pts) >= MIN_POINTS:
                prev_pts = new_pts; active = True
            else:
                metrics["lost"] += 1; active = False
                if len(new_pts) > 0: prev_pts = new_pts
        trail.append(good_new.reshape(-1, 2) if survived > 0 else np.empty((0, 2)))
        if len(trail) > TRAIL_LEN: trail.pop(0)
        dp = prev_pts if active else np.empty((0, 1, 2), dtype=np.float32)
        out.write(annotate(frame, bbox, dp, trail, fidx, label, active))
        metrics["frame"].append(fidx); metrics["n_pts"].append(survived)
        metrics["bbox"].append(bbox)
        metrics["centroid"].append(
            pts_to_centroid(prev_pts) if len(prev_pts) > 0 else metrics["centroid"][-1])
        prev_gray = curr_gray
        if fidx % 100 == 0:
            print(f"  [{label}] frame {fidx}/{n_total}  tracked_pts={survived}")
        fidx += 1
    cap.release(); out.release()
    tot = len(metrics["frame"]); lost = metrics["lost"]
    print(f"[{label}] Done — {fidx - start_frame} frames | "
          f"tracked {tot - lost}/{tot} ({100 * (tot - lost) / max(tot, 1):.1f}%) | saved: {output_path}")
    return metrics


# ── Color-guided tracker (for ball videos) ──────────────────────────────────
def detect_ball_by_color(frame, search_bbox, hsv_lower, hsv_upper,
                         min_area=300, exclude_regions=None):
    """
    Returns best-fit bounding box around the ball using HSV color mask.
    exclude_regions: list of (x,y,w,h) rectangles to mask out (e.g. the car).
    """
    x, y, w, h = search_bbox
    fh, fw = frame.shape[:2]
    x = max(0, x); y = max(0, y); w = min(w, fw - x); h = min(h, fh - y)
    roi = frame[y:y+h, x:x+w]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, hsv_lower, hsv_upper)
    if exclude_regions:
        for (ex, ey, ew, eh) in exclude_regions:
            ex2 = max(0, ex - x); ey2 = max(0, ey - y)
            ew2 = min(ew, w - ex2); eh2 = min(eh, h - ey2)
            if ew2 > 0 and eh2 > 0:
                mask[ey2:ey2+eh2, ex2:ex2+ew2] = 0
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best = None; best_area = 0
    for c in contours:
        area = cv2.contourArea(c)
        if area > min_area and area > best_area:
            bx, by, bw, bh = cv2.boundingRect(c)
            ar = max(bw, bh) / max(min(bw, bh), 1)
            if ar < 4:  # roughly circular/squarish
                best = (bx + x, by + y, bw, bh)
                best_area = area
    return best


def run_color_guided_tracker(input_path, output_path, init_roi, label,
                             hsv_lower, hsv_upper,
                             start_frame=0, exclude_regions=None,
                             search_pad=150, thresh_ratio=0.01,
                             block_size=3, min_dist=6):
    """
    Hybrid tracker: uses color detection to get a reliable bbox each frame,
    then seeds Harris+LK inside that bbox.
    Falls back to color detection when LK loses too many points.
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open: {input_path}")
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    fw     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh_v   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = cv2.VideoWriter(output_path,
                          cv2.VideoWriter_fourcc(*"mp4v"), fps, (fw, fh_v))
    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    ret, frame = cap.read()
    if not ret:
        cap.release(); out.release()
        raise RuntimeError("Cannot read first frame.")

    # Initialize with color detection on first frame
    color_bbox = detect_ball_by_color(
        frame, init_roi, hsv_lower, hsv_upper,
        exclude_regions=exclude_regions)
    if color_bbox is None:
        color_bbox = clamp(init_roi, frame.shape)
        print(f"[{label}] Warning: no color blob on first frame, using init_roi")

    prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    bbox = clamp(color_bbox, frame.shape)
    prev_pts = detect_harris_corners(prev_gray, bbox,
                                     block_size=block_size,
                                     thresh_ratio=thresh_ratio,
                                     min_dist=min_dist)
    if len(prev_pts) == 0:
        # seed from color bbox center
        cx, cy = bbox[0] + bbox[2]//2, bbox[1] + bbox[3]//2
        prev_pts = np.array([[cx, cy]], dtype=np.float32).reshape(-1, 1, 2)

    print(f"[{label}] Start frame={start_frame}, corners={len(prev_pts)}, bbox={bbox}")
    trail = []; fidx = start_frame; active = True
    metrics = {"frame": [], "n_pts": [], "bbox": [], "centroid": [], "lost": 0, "redet": 0}
    out.write(annotate(frame, bbox, prev_pts, [], fidx, label))
    metrics["frame"].append(fidx); metrics["n_pts"].append(len(prev_pts))
    metrics["bbox"].append(bbox); metrics["centroid"].append(pts_to_centroid(prev_pts))
    fidx += 1

    while True:
        ret, frame = cap.read()
        if not ret: break
        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        # 1) Try LK optical flow
        good_new, _ = track_lk(prev_gray, curr_gray, prev_pts)
        survived = len(good_new)

        # 2) Always run color detection in an expanded search window
        sx = max(0, bbox[0] - search_pad)
        sy = max(0, bbox[1] - search_pad)
        sw = min(fw - sx, bbox[2] + 2 * search_pad)
        sh = min(fh_v - sy, bbox[3] + 2 * search_pad)
        color_det = detect_ball_by_color(
            frame, (sx, sy, sw, sh), hsv_lower, hsv_upper,
            exclude_regions=exclude_regions)

        if survived >= MIN_POINTS:
            lk_bbox = clamp(pts_to_bbox(good_new), frame.shape)
            # Blend LK result with color detection when available
            if color_det is not None:
                # Use color bbox (more reliable for uniform-color ball)
                bbox = clamp(color_det, frame.shape)
                # Re-detect corners inside color bbox
                new_pts = detect_harris_corners(curr_gray, bbox,
                                                block_size=block_size,
                                                thresh_ratio=thresh_ratio,
                                                min_dist=min_dist)
                prev_pts = new_pts if len(new_pts) >= MIN_POINTS else good_new
            else:
                bbox = lk_bbox
                prev_pts = good_new
            active = True
        else:
            metrics["redet"] += 1
            if color_det is not None:
                bbox = clamp(color_det, frame.shape)
                new_pts = detect_harris_corners(curr_gray, bbox,
                                                block_size=block_size,
                                                thresh_ratio=thresh_ratio,
                                                min_dist=min_dist)
                if len(new_pts) >= MIN_POINTS:
                    prev_pts = new_pts; active = True
                else:
                    metrics["lost"] += 1; active = False
                    prev_pts = new_pts if len(new_pts) > 0 else prev_pts
            else:
                metrics["lost"] += 1; active = False

        trail.append(good_new.reshape(-1, 2) if survived > 0 else np.empty((0, 2)))
        if len(trail) > TRAIL_LEN: trail.pop(0)
        dp = prev_pts if active else np.empty((0, 1, 2), dtype=np.float32)
        out.write(annotate(frame, bbox, dp, trail, fidx, label, active))
        metrics["frame"].append(fidx); metrics["n_pts"].append(survived)
        metrics["bbox"].append(bbox)
        metrics["centroid"].append(
            pts_to_centroid(prev_pts) if len(prev_pts) > 0 else metrics["centroid"][-1])
        prev_gray = curr_gray
        if fidx % 100 == 0:
            print(f"  [{label}] frame {fidx}/{n_total}  tracked_pts={survived}")
        fidx += 1

    cap.release(); out.release()
    tot = len(metrics["frame"]); lost = metrics["lost"]
    print(f"[{label}] Done — {fidx - start_frame} frames | "
          f"tracked {tot - lost}/{tot} ({100 * (tot - lost) / max(tot, 1):.1f}%) | saved: {output_path}")
    return metrics


# ── Person tracker (white-clothing locked, no bbox bleed) ───────────────────
def run_person_tracker(input_path, output_path, init_roi, label,
                       start_frame=0, thresh_ratio=0.005,
                       block_size=3, min_dist=8,
                       max_bbox_size=None):
    """
    Tracks the white-clothed person.
    max_bbox_size=(w, h): hard cap on bounding box dimensions so the box never
    expands to engulf a different person entering the frame.
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open: {input_path}")
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    fw     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh_v   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out = cv2.VideoWriter(output_path,
                          cv2.VideoWriter_fourcc(*"mp4v"), fps, (fw, fh_v))
    if start_frame > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    ret, frame = cap.read()
    if not ret:
        cap.release(); out.release()
        raise RuntimeError("Cannot read first frame.")

    prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    bbox = clamp(init_roi, frame.shape)
    # Record initial box dimensions as the reference size
    init_w, init_h = bbox[2], bbox[3]
    if max_bbox_size is None:
        max_bbox_size = (int(init_w * 1.6), int(init_h * 1.4))

    prev_pts = detect_harris_corners(prev_gray, bbox,
                                     block_size=block_size,
                                     thresh_ratio=thresh_ratio,
                                     min_dist=min_dist)
    if len(prev_pts) == 0:
        cap.release(); out.release()
        raise RuntimeError(f"[{label}] No corners in ROI.")
    print(f"[{label}] Start frame={start_frame}, corners={len(prev_pts)}, roi={init_roi}")
    print(f"[{label}] Max bbox size capped at {max_bbox_size}")

    trail = []; fidx = start_frame; active = True
    metrics = {"frame": [], "n_pts": [], "bbox": [], "centroid": [], "lost": 0, "redet": 0}
    out.write(annotate(frame, bbox, prev_pts, [], fidx, label))
    metrics["frame"].append(fidx); metrics["n_pts"].append(len(prev_pts))
    metrics["bbox"].append(bbox); metrics["centroid"].append(pts_to_centroid(prev_pts))
    fidx += 1

    while True:
        ret, frame = cap.read()
        if not ret: break
        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Filter points: keep only those inside or near current bbox
        if len(prev_pts) > 0:
            bx, by, bw, bh = bbox
            pad_filter = 30
            in_box = ((prev_pts[:, 0, 0] >= bx - pad_filter) &
                      (prev_pts[:, 0, 0] <= bx + bw + pad_filter) &
                      (prev_pts[:, 0, 1] >= by - pad_filter) &
                      (prev_pts[:, 0, 1] <= by + bh + pad_filter))
            prev_pts = prev_pts[in_box]

        good_new, _ = track_lk(prev_gray, curr_gray, prev_pts)
        survived = len(good_new)

        if survived >= MIN_POINTS:
            raw_bbox = pts_to_bbox(good_new, pad=15)
            # Enforce size cap: preserve centroid, clamp dimensions
            rx, ry, rw, rh = raw_bbox
            rcx, rcy = rx + rw // 2, ry + rh // 2
            capped_w = min(rw, max_bbox_size[0])
            capped_h = min(rh, max_bbox_size[1])
            capped_x = rcx - capped_w // 2
            capped_y = rcy - capped_h // 2
            bbox = clamp((capped_x, capped_y, capped_w, capped_h), frame.shape)
            prev_pts = good_new; active = True
        else:
            metrics["redet"] += 1
            new_pts = detect_harris_corners(curr_gray, bbox,
                                            block_size=block_size,
                                            thresh_ratio=thresh_ratio,
                                            min_dist=min_dist)
            if len(new_pts) >= MIN_POINTS:
                prev_pts = new_pts; active = True
            else:
                metrics["lost"] += 1; active = False
                if len(new_pts) > 0: prev_pts = new_pts

        trail.append(good_new.reshape(-1, 2) if survived > 0 else np.empty((0, 2)))
        if len(trail) > TRAIL_LEN: trail.pop(0)
        dp = prev_pts if active else np.empty((0, 1, 2), dtype=np.float32)
        out.write(annotate(frame, bbox, dp, trail, fidx, label, active))
        metrics["frame"].append(fidx); metrics["n_pts"].append(survived)
        metrics["bbox"].append(bbox)
        metrics["centroid"].append(
            pts_to_centroid(prev_pts) if len(prev_pts) > 0 else metrics["centroid"][-1])
        prev_gray = curr_gray
        if fidx % 100 == 0:
            print(f"  [{label}] frame {fidx}/{n_total}  tracked_pts={survived}")
        fidx += 1

    cap.release(); out.release()
    tot = len(metrics["frame"]); lost = metrics["lost"]
    print(f"[{label}] Done — {fidx - start_frame} frames | "
          f"tracked {tot - lost}/{tot} ({100 * (tot - lost) / max(tot, 1):.1f}%) | saved: {output_path}")
    return metrics


def plot_metrics(metrics, title):
    frames = np.array(metrics["frame"]); n_pts = np.array(metrics["n_pts"])
    cents  = np.array(metrics["centroid"]); boxes = np.array(metrics["bbox"])
    fig, ax = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    fig.suptitle(title, fontsize=14)
    ax[0].plot(frames, n_pts, color="steelblue", lw=0.8)
    ax[0].axhline(MIN_POINTS, color="red", ls="--", label=f"MIN={MIN_POINTS}")
    ax[0].set_ylabel("Tracked pts"); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[1].plot(frames, cents[:, 0], label="cx", color="tomato", lw=0.8)
    ax[1].plot(frames, cents[:, 1], label="cy", color="seagreen", lw=0.8)
    ax[1].set_ylabel("Centroid (px)"); ax[1].legend(); ax[1].grid(alpha=0.3)
    ax[2].plot(frames, boxes[:, 2] * boxes[:, 3], color="darkorchid", lw=0.8)
    ax[2].set_ylabel("BBox area (px²)"); ax[2].set_xlabel("Frame"); ax[2].grid(alpha=0.3)
    plt.tight_layout(); plt.show()


def preview_video(path, title, n=5):
    cap = cv2.VideoCapture(path); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = [int(total * i / (n - 1)) for i in range(n)]; idxs[-1] = min(idxs[-1], total - 1)
    fig, axes = plt.subplots(1, n, figsize=(20, 4)); fig.suptitle(title, fontsize=13)
    for ax, fi in zip(axes, idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi); ret, f = cap.read()
        if ret: ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        ax.set_title(f"frame {fi}", fontsize=8); ax.axis("off")
    cap.release(); plt.tight_layout(); plt.show()


print("✅ All helpers loaded. Ready to track.")


✅ All helpers loaded. Ready to track.


---
## Video 1 — Thrown Ball
`19-May-2026_at_2_12_PM.mov` → `output_v1_ball.mp4`

**Fix:** The previous code tracked the **still blue ball** sitting against the left wall.
The correct target is the **thrown ball** — a nearly-black ball that enters from near
the car (right side) and rolls across the courtyard.

Changes made:
- `start_frame=3`: thrown ball first appears at frame 3 (before this, only the still ball is in scene)
- `V1_ROI=(684, 704, 140, 140)`: correct ROI around the thrown ball in frame 3 (center ~754,774)
- HSV range widened and V lowered to capture the near-black thrown ball: `V∈[0,80]`
- `V1_EXCLUDE` now masks out both the car AND the still ball (left wall area)
  so color re-detection never snaps back to the stationary ball.
- `search_pad=200` to follow the ball as it rolls across the full width of the courtyard.


In [2]:
V1_IN  = "19-May-2026 at 2_12 PM.mov"
V1_OUT = "output_v1_ball.mp4"

# ── start_frame: thrown ball first enters at frame 3 (near the car, right side)
V1_START = 3

# ── Corrected ROI: around the THROWN ball in frame 3 (center≈754,774 r≈70)
# The still blue ball on the left wall (ROI≈100,770) is intentionally excluded below.
V1_ROI = (684, 704, 140, 140)

# HSV range for the thrown ball (nearly pure black/very dark)
V1_HSV_LOW  = np.array([0,   0,   0])
V1_HSV_HIGH = np.array([180, 80, 80])

# Exclude regions:
#   1) Car (upper-right dark area): x=600-1080, y=300-750
#   2) Still ball (left-wall blue ball): x=60-230, y=760-900
#      — prevents color re-detection from locking onto the stationary ball
V1_EXCLUDE = [
    (600, 300, 480, 450),   # car
    (60,  760, 170, 140),   # still ball (stationary — not the target)
]

metrics_v1 = run_color_guided_tracker(
    V1_IN, V1_OUT,
    init_roi    = V1_ROI,
    label       = "V1-Ball",
    hsv_lower   = V1_HSV_LOW,
    hsv_upper   = V1_HSV_HIGH,
    start_frame = V1_START,
    exclude_regions = V1_EXCLUDE,
    search_pad  = 200,
    thresh_ratio = 0.01,
    block_size  = 3,
    min_dist    = 5,
)


[V1-Ball] Warning: no color blob on first frame, using init_roi
[V1-Ball] Start frame=3, corners=14, bbox=(684, 704, 140, 140)
  [V1-Ball] frame 100/177  tracked_pts=12
[V1-Ball] Done — 174 frames | tracked 173/174 (99.4%) | saved: output_v1_ball.mp4


In [ ]:
plot_metrics(metrics_v1, "V1 Ball — Tracking Metrics")
preview_video(V1_OUT, "V1 Ball — Output Sample Frames")


---
## Video 2 — Ball Rolling (courtyard)
`19-May-2026_at_2_13_PM.mov` → `output_v2_ball.mp4`

**Fix:** The original ROI `(330, 1820, 200, 100)` was correct in concept, but Harris corners alone on the ball's smooth surface lost tracking quickly. A **color-guided hybrid tracker** ensures the box stays on the ball as it rolls toward the camera and grows larger in frame.

- Ball enters from the very bottom at frame 34 (`x≈360, y≈1857`).
- It rolls toward the camera, grows from ~90px to ~200px diameter.


In [ ]:
V2_IN    = "19-May-2026_at_2_13_PM.mov"
V2_OUT   = "output_v2_ball.mp4"
V2_START = 34   # ball first visible at bottom of frame

# ROI at frame 34: ball just entering at bottom
V2_ROI = (330, 1820, 260, 100)

# Same dark-blue ball
V2_HSV_LOW  = np.array([95,  40,  10])
V2_HSV_HIGH = np.array([145, 255, 90])

# Exclude the car (right side, rows 0-700)
V2_EXCLUDE = [(550, 0, 530, 700)]

metrics_v2 = run_color_guided_tracker(
    V2_IN, V2_OUT,
    init_roi    = V2_ROI,
    label       = "V2-Ball",
    hsv_lower   = V2_HSV_LOW,
    hsv_upper   = V2_HSV_HIGH,
    start_frame = V2_START,
    exclude_regions = V2_EXCLUDE,
    search_pad  = 200,   # larger pad because ball moves fast toward camera
    thresh_ratio = 0.01,
    block_size  = 3,
    min_dist    = 5,
)


In [ ]:
plot_metrics(metrics_v2, "V2 Ball — Tracking Metrics")
preview_video(V2_OUT, "V2 Ball — Output Sample Frames")


---
## Video 3 — Person Walking
`19-May-2026_at_2_15_PM.mov` → `output_v3_person.mp4`

**Fixes applied:**
1. **Corrected ROI** in frame 0: the white-shirt person is at `x=390, y=500, w=190, h=380` (body only, not shadow).
2. **Bounding-box size cap**: the box is hard-capped at `1.6× initial width` and `1.4× initial height` — this prevents the tracker from expanding to engulf a second person walking into frame.
3. **Lower `thresh_ratio=0.005`** so Harris finds corners on plain white fabric.
4. **Point filtering**: only points inside (or just outside) the current bbox are tracked, so corners on a passing person are discarded.


In [ ]:
V3_IN  = "19-May-2026_at_2_15_PM.mov"
V3_OUT = "output_v3_person.mp4"

# ── Corrected ROI: white-shirt person body in frame 0
# Person head ~y=500, feet ~y=870, body width ~190px centred at x~480
V3_ROI = (385, 500, 195, 380)

metrics_v3 = run_person_tracker(
    V3_IN, V3_OUT,
    init_roi     = V3_ROI,
    label        = "V3-Person",
    thresh_ratio = 0.005,
    block_size   = 3,
    min_dist     = 7,
    # Hard cap: no wider than ~310px, no taller than ~530px
    # (person starts ~195×380, allow 1.6× / 1.4×)
    max_bbox_size = (310, 530),
)


In [ ]:
plot_metrics(metrics_v3, "V3 Person — Tracking Metrics")
preview_video(V3_OUT, "V3 Person — Output Sample Frames")


---
## Video 4 — Bird in Sky (Eagle/Hawk)
`19-May-2026_at_2_16_PM.MOV` → `output_v4_bird.mp4`

**Fix:** The original ROI `(515, 70, 90, 60)` targeted the tiny bird near the upper wires. The actual eagle/hawk the user wants is flying in the **open sky**, well below the wire cluster.

In frame 0 the eagle is at approximately `x=495, y=440, w=70, h=60` (video is 720×1280 portrait).

Tuning:
- `block_size=2`, very low `thresh_ratio=0.001` — bird is a tiny dark shape against grey sky.
- `min_dist=3` — allows many close-together corners on the small silhouette.
- `search_pad=100` — allows the tracker to re-find the bird as it soars.


In [ ]:
V4_IN  = "19-May-2026_at_2_16_PM.MOV"
V4_OUT = "output_v4_bird.mp4"

# ── Corrected ROI: eagle/hawk in OPEN SKY, below the wires (not near wire at top)
# Frame 0: bird visible at approx x=495, y=440, w=70, h=60  (720×1280 video)
V4_ROI = (455, 420, 100, 80)

metrics_v4 = run_tracker(
    V4_IN, V4_OUT, V4_ROI,
    label        = "V4-Bird",
    thresh_ratio = 0.001,   # extremely sensitive — bird is a subtle dark speck
    block_size   = 2,       # tiny neighbourhood for a tiny target
    min_dist     = 3,
)


In [ ]:
plot_metrics(metrics_v4, "V4 Bird — Tracking Metrics")
preview_video(V4_OUT, "V4 Bird — Output Sample Frames")


---
## Summary

In [ ]:
results = [
    ("V1 Ball",   V1_OUT, metrics_v1),
    ("V2 Ball",   V2_OUT, metrics_v2),
    ("V3 Person", V3_OUT, metrics_v3),
    ("V4 Bird",   V4_OUT, metrics_v4),
]
print(f"{'Video':<12} {'Frames':>7} {'Tracked%':>9} {'Avg pts':>8}  Output")
print("-" * 60)
for name, path, m in results:
    tot  = len(m["frame"])
    lost = m["lost"]
    avg  = np.mean(m["n_pts"])
    pct  = 100 * (tot - lost) / max(tot, 1)
    print(f"{name:<12} {tot:>7d} {pct:>8.1f}%  {avg:>7.1f}  {path}")
